# ML-04 — Search Intelligence Data Contract (Lane 2: Content Opportunity Scoring)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lakes41/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook writes (and then **queries to verify**) the data contract for Lane 2: the Content Opportunity Scoring queue. Sections **in order** — each contract claim gets a SQL/query proof right after it.

**Backend:** DuckDB over the Hugging Face warehouse release (`FlyRank/internship-warehouse`). If `HF_TOKEN` is not available in your environment the first time you open this, follow the 2-minute setup printed in cell 2, then Run All again. (The queries also run against the starter CSV as a fallback, so the structure is visible even without the warehouse.)

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-data-contracts` + `flyrank/flyrank-data` for this task.


## 0. Setup: DuckDB + Hugging Face auth

Run this cell first. It installs any missing package, reads your `HF_TOKEN` (env to Colab Secret to prompt), and registers the warehouse tables with DuckDB.


In [1]:
# Setup -- one-shot install, token, and table registration.
import importlib, subprocess, sys, os, getpass, textwrap

def ensure_pkg(name, pip_name=None):
    try:
        importlib.import_module(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name or name])

ensure_pkg("duckdb")
ensure_pkg("pandas")
import duckdb, pandas as pd, numpy as np

# --- Token -----------------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata  # type: ignore
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
# Prompt (last resort) -- disabled when the env has no terminal (Colab Secrets / env var is the right path).
if not HF_TOKEN and sys.stdin.isatty():
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

# --- DuckDB + tables -------------------------------------------------------
# DuckDB needs a writable extensions cache directory for httpfs. Try to create one.
import tempfile as _tmp
_ext_dir = os.environ.get("DUCKDB_EXTENSION_DIR") or os.path.join(os.path.expanduser("~"), ".duckdb", "extensions")
try:
    os.makedirs(_ext_dir, exist_ok=True)
    # Test write access
    with open(os.path.join(_ext_dir, ".write_test"), "w") as _fh:
        _fh.write("ok")
    try:
        os.remove(os.path.join(_ext_dir, ".write_test"))
    except Exception:
        pass
except Exception:
    _ext_dir = _tmp.mkdtemp(prefix="duckdb_ext_")
    os.environ["DUCKDB_EXTENSION_DIR"] = _ext_dir

con = duckdb.connect()
USING_WAREHOUSE = False
TABLES = {}

if HF_TOKEN:
    try:
        con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
        REL = "hf://datasets/FlyRank/internship-warehouse"
        TABLES = {
            "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
            "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
            "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
            "fact_daily_month":  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
            "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
        }
        # Quick metadata probe to confirm connectivity (Parquet metadata only, no data download)
        _rowcounts = {t: con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
                      for t, src in TABLES.items()}
        print("WAREHOUSE CONNECTED  (build v20260703):")
        for t, n in _rowcounts.items():
            print(f"  {t:20s} {n:>12,} rows")
        USING_WAREHOUSE = True
    except Exception as exc:
        print(f"WAREHOUSE FAILED ({exc.__class__.__name__}: {exc}). Falling back to starter CSV.")
else:
    print(textwrap.dedent("""
    HF_TOKEN not set. Running with the STARTER CSV FALLBACK.
    To run on the real full 79-million-row warehouse:
      1. Create a free HF account at huggingface.co/join
      2. Accept the terms (instant!) at huggingface.co/datasets/FlyRank/internship-warehouse
      3. Create a READ token at huggingface.co/settings/tokens
      4. Re-run this notebook with  export HF_TOKEN=hf_...   in your shell,
         or paste it into the Colab Secrets panel as HF_TOKEN, then Run All.
    """))

# --- Starter-CSV fallback (always present; used only when warehouse is missing) ---
STARTER_CSV = "../../data/raw/content_refresh_anonymized.csv"
assert os.path.exists(STARTER_CSV), f"starter CSV missing: {STARTER_CSV}"
starter = pd.read_csv(STARTER_CSV)
con.register("starter_df", starter)
print(f"starter_df registered: {len(starter):,} rows (fallback always available)")

from sklearn.preprocessing import MinMaxScaler  # reused in later cells


WAREHOUSE FAILED (Error: An error occurred while trying to automatically install the required extension 'httpfs':
Failed to create directory "/Users/amiroyeleke/.duckdb": Operation not permitted). Falling back to starter CSV.


starter_df registered: 30,000 rows (fallback always available)


## 1. The contract, in plain words (5 answers)

For Lane 2 -- Content Opportunity Scoring (the editor's review-first queue):

| # | Question | Answer |
|---|----------|--------|
| 1 | **What does one row mean?** | One row = **one content item (one page)**, with metrics aggregated over a fixed trailing-90-day feature window ending the day *before* the label window starts. The unique key is `content_hash_id` (warehouse) / `content_id` (starter). |
| 2 | **Which table(s) will you use?** | Primary: `fact_content_daily_performance` (daily aggregates for feature windows and the forward label). Joined to `dim_content` (content metadata: word count, keyword context) and `dim_clients` (filtering out clients without GA4/GSC flags for the window). Optional enrichment from `fact_content_query_90d` (query concentration, rare-share) after window-alignment leakage check. |
| 3 | **Which time window?** | Iteration partition for this notebook: **`month=2026-03`** (mid-panel -- explicitly NOT the last month, which is leakage-land). For each content item, *features* use `2025-12-02 to 2026-02-28` (trailing 90 days *before* March). *Label* uses `2026-03-01 to 2026-03-31` (the forward March outcome -- did March impressions drop >20% vs prev 30d?). |
| 4 | **What do you predict/rank?** | A continuous **opportunity_score proxy** = weighted combination of: (a) severity of the *forward* decline observed in March (the label, not a feature), (b) log-impressions visibility in feature window, (c) striking-distance position on a high-volume keyword. For ranking: higher score = editor reviews this page first. |
| 5 | **One thing you deliberately EXCLUDE?** | I **exclude `trend_direction` / `trend_pct` / any bucket derived from the last-30-vs-prev-30 impression comparison** as a feature. Those are label-source columns (the target is built from them in the starter CSV; in the warehouse their equivalents are the forward window). Using them as input is training a model to reproduce the pipeline's own rule. |

> **In one sentence (the framing-paragraph from the skill):** *For content editors, deciding which 200 pages to refresh this sprint, we will build a scored priority queue from 90-day trailing features plus content metadata plus query mix, ranking by an opportunity proxy whose decline leg is measured in the 30 days AFTER the feature window and scored using precision@200 on a held-out client split. A wrong call costs an editor's hour on a dead page OR missed recovery on a high-visibility page. A plain rule is not enough because the opportunity is the product of three different tangled signals (decline x visibility x upside) and missingness follows content-type patterns. We will claim only decision-support directional results observed on historical data.*


In [2]:
# Cell reserved for the Section 1 markdown. (Contract prose goes ABOVE.)
# The three verification queries that PROVE these 5 contract claims live in Section 3, below.
print("Contract 5-answers read. Verification queries begin in the next section (S3).")


Contract 5-answers read. Verification queries begin in the next section (S3).


## 2. Fields: feature / label / context / excluded

Every field I plan to touch. One bucket per line -- no field appears twice.

### Features (knowable BEFORE the decision moment; safe to learn from)
| Feature name | Source table / window | Why it is a feature |
|---|---|---|
| `log_impressions_trail90` | `fact_content_daily_performance`, 90 days *before* label month | log(1+sum of search impressions) for the page in the 90-day feature window only. |
| `ctr_trail90` | `fact_content_daily_performance`, 90 days before label month | Click-through-rate (100 x clicks_trail90 / impressions_trail90), same 90-day window. |
| `position_trail90` | `fact_content_daily_performance`, 90 days before label month | Volume-weighted mean GSC position across the 90-day feature window. |
| `content_age_days` | `dim_content` (age = feature-window-end minus creation date) | Staleness proxy (static per item, always knowable). |
| `has_word_count` + `word_count` (when present) | `dim_content` (last-measured before window end) | Content-depth signal; missingness modeled via flag. |

### Label / proxy (what I predict -- NEVER a feature)
| Label name | Source | Why it is a label |
|---|---|---|
| `forward_decline_severity` + `is_declining_forward` (binary) | `fact_content_daily_performance`, label month `2026-03` vs prev-30 (`2026-02-01 to 2026-02-28`) | Observed OUTCOME that happened AFTER the feature window. This is what the model tries to reproduce, not what it is allowed to see. |
| `opportunity_score` (the rank key) | Composite: `0.4 * forward_decline_severity + 0.4 * log(impressions_trail90+1) + 0.2 * striking_headroom` | The proxy that defines "review first"; its decline leg comes only from the forward outcome, so the model never sees the decline_severity leg itself. |

### Context (used only for grouping / splitting / joins -- never fed to the model)
| Field | Why it is context |
|---|---|
| `content_hash_id`, `client_hash_id` | Joins and group-keys. `client_hash_id` is used for per-client train/test splits (the honest split). |
| `gsc_data_available`, `ga4_data_available` | Three-valued filters (IS TRUE / NOT TRUE) -- dropped after filtering. |
| `access_profile` (dim_clients) | Reading only; not learnable. |

### Excluded (deliberately refused, each with a why)
| Field | Why excluded |
|---|---|
| `trend_pct`, `trend_direction`, `is_declining_label` (starter CSV equivalents) | LABEL-SOURCE columns. Using them is learning the pipeline's own rule, not the world. |
| `impressions_last30` / `clicks_last30` from the starter 30d-vs-30d buckets | Same as above -- the last-30 of the starter CSV is the label period. Only prev-30 is feature-safe. |
| `fact_content_query_90d.*_90d` totals (window overlap) | The 90d query table ends in June 2026. If your label lives in June 2026 these are leakage. For this notebook's March-window they are provably disjoint, but I exclude all per-query `*_90d` totals from the initial 5-feature set to keep the contract simple and auditable. |
| `provider_used`, `model_used` (LLM origin fields in starter) | Product-decision flags: the action is an editorial refresh and should not prioritize pages based on which LLM generated them. |


In [3]:
# Fields-classification prose goes ABOVE this cell.
# The 5-feature frame with per-feature "knowable-when" lines is built in S3.
print("Field classification read. Verification queries are next.")


Field classification read. Verification queries are next.


## 3. Verify it with queries (3 contract checks), then 5-feature frame, then the deliberate-leakage trap

Every claim in sections 1-2 gets a query. I run exactly **three verification queries** as required by the card:
1. **Grain probe** -- one row really is one content item (GROUP BY key, HAVING count > 1 -> expect 0 hits).
2. **Slice row count + date span** -- how many content items survive the lane filter in `month=2026-03`, and what is the MIN/MAX `report_date` feeding the feature window.
3. **Availability with `IS TRUE`** -- three-valued `gsc_data_available` filter; show rows before, after `IS TRUE`, and the survival count with `ga4_data_available IS TRUE` stacked on top.

Then (S3b): build the five-feature frame from the same month, one line per feature explaining "*knowable at the decision moment because...*".

Then (S3c): the **deliberate-leakage trap**. Add ONE label-derived column on purpose, watch the prec@200 jump, then delete it and keep the honest number.

### 3a. Three verification queries (grain, counts+span, availability)

**Important note on the data source:** If the Hugging Face warehouse is connected (`USING_WAREHOUSE = True`), the queries below hit `month=2026-03` of `fact_content_daily_performance` -- the full-warehouse contract proof. If the warehouse is not connected and we fall back to `starter_df`, the exact same logic runs with the starter CSV's feature/label equivalents and we note the counts next to each expected warehouse number.


In [4]:
# ===== QUERY 1 of 3: GRAIN PROBE (one row really IS one content item) =====
#    Expected: 0 duplicate keys. Grain passes if the query returns 0 rows.

if USING_WAREHOUSE:
    # Warehouse grain for our 90-day feature then 1-month label per-content aggregates:
    # First build the per-content feature+label frame for month=2026-03, then probe its key.
    grain_sql = f"""
    WITH feature_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions)        AS imp_trail90,
            COUNT(DISTINCT report_date)  AS days_seen,
            COUNT(*)                     AS daily_rows
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN DATE '2025-12-02' AND DATE '2026-02-28'
          AND month = ANY (LIST_VALUE('2025-12', '2026-01', '2026-02'))
        GROUP BY 1, 2
        HAVING imp_trail90 >= 100
    )
    SELECT client_hash_id, content_hash_id, COUNT(*) AS n
    FROM feature_agg
    GROUP BY 1, 2
    HAVING n > 1
    LIMIT 5
    """
    dupes = con.sql(grain_sql).df()
    print("QUERY 1 (warehouse): GRAIN PROBE on feature-agg (client x content per content item)")
    print(f"  duplicate keys found: {len(dupes)}   (expected 0 -> grain = one row per content item PASSED)")
    if len(dupes):
        print(dupes)
else:
    # Fallback: starter CSV grain = content_id unique
    grain = con.sql("""
        SELECT content_id, COUNT(*) AS n
        FROM starter_df GROUP BY content_id HAVING n > 1 LIMIT 5
    """).df()
    print("QUERY 1 (starter CSV fallback): GRAIN PROBE")
    print(f"  rows = {len(starter):,}, unique content_ids = {starter['content_id'].nunique():,}")
    print(f"  duplicate content_id rows: {len(grain)}   (expected 0 -> grain = one row = one page PASSED)")
    if len(grain):
        print(grain)


QUERY 1 (starter CSV fallback): GRAIN PROBE
  rows = 30,000, unique content_ids = 30,000
  duplicate content_id rows: 0   (expected 0 -> grain = one row = one page PASSED)


In [5]:
# ===== QUERY 2 of 3: ROW COUNT + DATE SPAN for the lane slice in month=2026-03 =====
if USING_WAREHOUSE:
    q2 = f"""
    WITH
      feat AS (
        SELECT
            client_hash_id, content_hash_id,
            MIN(report_date)    AS feat_min_d,
            MAX(report_date)    AS feat_max_d,
            SUM(gsc_impressions) AS imp_trail90,
            SUM(gsc_clicks)      AS clk_trail90
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN DATE '2025-12-02' AND DATE '2026-02-28'
        GROUP BY 1, 2
      ),
      lbl AS (
        SELECT DISTINCT client_hash_id, content_hash_id
        FROM {TABLES['fact_daily_month']}
        WHERE gsc_impressions > 0
      ),
      dc AS (
        SELECT content_hash_id, word_count, keyword_volume_estimate AS search_volume
        FROM {TABLES['dim_content']}
      )
    SELECT
        COUNT(*)                                          AS row_count,
        MIN(f.feat_min_d)                                 AS feature_span_min,
        MAX(f.feat_max_d)                                 AS feature_span_max,
        SUM(CASE WHEN d.search_volume IS NOT NULL THEN 1 ELSE 0 END) AS n_with_keyword_data,
        SUM(CASE WHEN d.word_count    IS NOT NULL THEN 1 ELSE 0 END) AS n_with_word_count
    FROM feat f
    JOIN lbl l USING (client_hash_id, content_hash_id)
    LEFT JOIN dc USING (content_hash_id)
    WHERE f.imp_trail90 >= 100
    """
    counts = con.sql(q2).df().iloc[0]
    print("QUERY 2 (warehouse): LANE SLICE ROWS + DATE SPAN")
    print(f"  content items in slice:              {counts['row_count']:>10,}")
    print(f"  feature window span:                 {counts['feature_span_min']}  ->  {counts['feature_span_max']}")
    print(f"  with keyword data (non-null sv):     {counts['n_with_keyword_data']:>10,} ({counts['n_with_keyword_data']/counts['row_count']:.0%})")
    print(f"  with word_count measured:            {counts['n_with_word_count']:>10,} ({counts['n_with_word_count']/counts['row_count']:.0%})")
else:
    lane = con.sql("""
        SELECT COUNT(*)                              AS n,
               MIN(content_age_days)                 AS age_min,
               MAX(content_age_days)                 AS age_max,
               AVG(impressions_90d)                  AS avg_imp,
               SUM(CASE WHEN search_volume IS NOT NULL AND search_volume > 0 THEN 1 ELSE 0 END) AS n_sv,
               SUM(CASE WHEN word_count    IS NOT NULL THEN 1 ELSE 0 END)                     AS n_wc
        FROM starter_df
        WHERE impressions_90d >= 100
          AND NOT (avg_position = 0 AND impressions_90d < 500)
    """).df().iloc[0]
    print("QUERY 2 (starter CSV fallback): LANE SLICE ROWS + CONTENT-AGE RANGE")
    print(f"  content items in slice:              {lane['n']:>10,}")
    print(f"  content_age_days range:              {lane['age_min']:.0f} d  ->  {lane['age_max']:.0f} d")
    print(f"  avg 90-day impressions (slice):         {lane['avg_imp']:>10,.0f}")
    print(f"  with search_volume>0 data:           {lane['n_sv']:>10,} ({lane['n_sv']/lane['n']:.0%})")
    print(f"  with word_count measured:            {lane['n_wc']:>10,} ({lane['n_wc']/lane['n']:.0%})")
    print("  (date span not applicable to snapshot CSV; see warehouse output when HF_TOKEN set.)")


QUERY 2 (starter CSV fallback): LANE SLICE ROWS + CONTENT-AGE RANGE
  content items in slice:                22,006.0
  content_age_days range:              90 d  ->  564 d
  avg 90-day impressions (slice):              7,081
  with search_volume>0 data:             12,907.0 (59%)
  with word_count measured:              15,451.0 (70%)
  (date span not applicable to snapshot CSV; see warehouse output when HF_TOKEN set.)


In [6]:
# ===== QUERY 3 of 3: AVAILABILITY with IS TRUE (3-valued flag handling) =====
if USING_WAREHOUSE:
    q3 = f"""
    WITH
      daily_mar AS (
        SELECT * FROM {TABLES['fact_daily_month']}
      ),
      clients AS (
        SELECT client_hash_id, gsc_data_start, ga4_data_start
        FROM {TABLES['dim_clients']}
      ),
      counts_per_client AS (
        SELECT
            d.client_hash_id,
            COUNT(*)                                          AS daily_rows,
            COUNT(*) FILTER (WHERE d.gsc_data_available IS TRUE)      AS gsc_true_rows,
            COUNT(*) FILTER (WHERE d.ga4_data_available IS TRUE)      AS ga4_true_rows,
            COUNT(*) FILTER (WHERE d.gsc_data_available IS NOT TRUE)  AS gsc_not_true_rows,
            COUNT(*) FILTER (WHERE d.ga4_data_available IS NOT TRUE)  AS ga4_not_true_rows
        FROM daily_mar d
        GROUP BY 1
      )
    SELECT
        COUNT(*)                                                     AS clients_total,
        SUM(CASE WHEN c.gsc_data_start IS NOT NULL THEN 1 ELSE 0 END) AS clients_with_gsc_start,
        SUM(cpc.gsc_true_rows)                                       AS gsc_true_rows_total,
        SUM(cpc.gsc_not_true_rows)                                   AS gsc_not_true_rows_total,
        SUM(cpc.ga4_true_rows)                                       AS ga4_true_rows_total,
        SUM(cpc.ga4_not_true_rows)                                   AS ga4_not_true_rows_total
    FROM counts_per_client cpc
    LEFT JOIN clients c USING (client_hash_id)
    """
    av = con.sql(q3).df().iloc[0]
    print("QUERY 3 (warehouse): AVAILABILITY -- three-valued flags with IS TRUE (month=2026-03)")
    print(f"  client count (appearing in March 2026):        {av['clients_total']:>6,}")
    print(f"  clients with gsc_data_start in dim_clients:    {av['clients_with_gsc_start']:>6,}")
    print(f"  daily rows, gsc_data_available IS TRUE:        {av['gsc_true_rows_total']:>12,}")
    print(f"  daily rows, gsc_data_available NOT TRUE:       {av['gsc_not_true_rows_total']:>12,}  (FALSE/NULL -- DROPPED)")
    print(f"  daily rows, ga4_data_available IS TRUE:        {av['ga4_true_rows_total']:>12,}")
    print(f"  daily rows, ga4_data_available NOT TRUE:       {av['ga4_not_true_rows_total']:>12,}  (FALSE/NULL -- DROPPED)")
    gsc_survive_pct = av['gsc_true_rows_total']/(av['gsc_true_rows_total']+av['gsc_not_true_rows_total']+1e-9)
    print(f"  -> GSC rows surviving IS TRUE filter: {gsc_survive_pct:.0%}")
else:
    q3s = con.sql("""
        SELECT
            COUNT(*)                                              AS total_rows,
            SUM(CASE WHEN avg_position IS NOT NULL AND avg_position <> 0 THEN 1 ELSE 0 END)    AS pos_true_rows,
            SUM(CASE WHEN avg_position IS NULL     OR  avg_position =  0 THEN 1 ELSE 0 END)    AS pos_not_true_rows,
            SUM(CASE WHEN word_count   IS NOT NULL THEN 1 ELSE 0 END)                            AS wc_true_rows,
            SUM(CASE WHEN word_count   IS NULL     THEN 1 ELSE 0 END)                            AS wc_not_true_rows,
            SUM(CASE WHEN search_volume IS NOT NULL AND search_volume > 0 THEN 1 ELSE 0 END)     AS sv_true_rows,
            SUM(CASE WHEN search_volume IS NULL     OR  search_volume = 0 THEN 1 ELSE 0 END)     AS sv_not_true_rows
        FROM starter_df
        WHERE impressions_90d >= 100
          AND NOT (avg_position = 0 AND impressions_90d < 500)
    """).df().iloc[0]
    print("QUERY 3 (starter CSV fallback): AVAILABILITY with IS-TRUE pattern")
    print(f"  lane rows total:                            {q3s['total_rows']:>8,}")
    print(f"  position data IS TRUE   (pos!=0 & not null): {q3s['pos_true_rows']:>8,}  ({q3s['pos_true_rows']/q3s['total_rows']:.0%})")
    print(f"  position data NOT TRUE (pos=0 | null):      {q3s['pos_not_true_rows']:>8,}  ({q3s['pos_not_true_rows']/q3s['total_rows']:.0%}) -- DROPPED")
    print(f"  word_count   IS TRUE:                       {q3s['wc_true_rows']:>8,}  ({q3s['wc_true_rows']/q3s['total_rows']:.0%})")
    print(f"  word_count   NOT TRUE (NULL):               {q3s['wc_not_true_rows']:>8,}  ({q3s['wc_not_true_rows']/q3s['total_rows']:.0%}) -- impute via flag")
    print(f"  search_volume IS TRUE:                      {q3s['sv_true_rows']:>8,}  ({q3s['sv_true_rows']/q3s['total_rows']:.0%})")
    print(f"  search_volume NOT TRUE:                     {q3s['sv_not_true_rows']:>8,}  ({q3s['sv_not_true_rows']/q3s['total_rows']:.0%}) -- impute via flag")
    print("  (three-valued gsc/ga4 flags live in the warehouse daily fact table; see warehouse output when HF_TOKEN set.)")


QUERY 3 (starter CSV fallback): AVAILABILITY with IS-TRUE pattern
  lane rows total:                            22,006.0
  position data IS TRUE   (pos!=0 & not null): 22,006.0  (100%)
  position data NOT TRUE (pos=0 | null):           0.0  (0%) -- DROPPED
  word_count   IS TRUE:                       15,451.0  (70%)
  word_count   NOT TRUE (NULL):                6,555.0  (30%) -- impute via flag
  search_volume IS TRUE:                      12,907.0  (59%)
  search_volume NOT TRUE:                      9,099.0  (41%) -- impute via flag
  (three-valued gsc/ga4 flags live in the warehouse daily fact table; see warehouse output when HF_TOKEN set.)


### 3b. Five-feature feature frame (max 5), from the same month

Each row of this frame is one content item. Each feature has one line: **"knowable at the decision moment because..."**.

| # | Feature | Type | Knowable at the decision moment because ... |
|---|---------|------|--------------------------------------------|
| 1 | `log_impressions_trail90` | numeric | ...it is summed only from `fact_content_daily_performance` rows in the 90 days *ending the day before* the label-month begins (2025-12-02 to 2026-02-28). An editor deciding on 2026-03-01 has this full 90-day history already. |
| 2 | `ctr_trail90` | numeric (rate, times 100) | ...it is `clicks_trail90 / impressions_trail90` using exactly the same 90-day-pre-March feature window. Both numerator and denominator are historical observations already recorded before 2026-03-01. |
| 3 | `position_trail90` (volume-weighted mean) | numeric | ...it is the impression-weighted average of `gsc_avg_position` over the 90-day feature window. Position data is per-day GSC output, all before the label month. |
| 4 | `content_age_days` | numeric | ...age = `2026-02-28 minus content_creation_date`. Creation date is a static property of each content item; by 2026-03-01 we have always known it. |
| 5 | `has_word_count` (plus `word_count` when present + missingness flag) | binary + numeric | ...word-count is a content-property measurement that is done *at publish time* or last-updated time; it never uses forward-label data. When NULL we carry a `has_word_count=0` flag instead of filling with 0 (which would silently encode content-type). |


In [7]:
# Build the FIVE-FEATURE frame from the same data source as queries 1-3.
if USING_WAREHOUSE:
    feat_frame_sql = f"""
    WITH
      feat90 AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_impressions)                                      AS imp_trail90,
            SUM(gsc_clicks)                                          AS clk_trail90,
            SUM(gsc_impressions * gsc_avg_position)
                / NULLIF(SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions END), 0)  AS pos_vw_trail90,
            COUNT(DISTINCT report_date)                              AS days_seen
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN DATE '2025-12-02' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING imp_trail90 >= 100
      ),
      dim_c AS (
        SELECT content_hash_id, word_count, created_at
        FROM {TABLES['dim_content']}
      )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        LN(1 + f.imp_trail90)                                         AS log_impressions_trail90,
        100.0 * f.clk_trail90 / NULLIF(f.imp_trail90, 0)             AS ctr_trail90,
        COALESCE(f.pos_vw_trail90, 0)                                 AS position_trail90,
        DATE_DIFF('DAY', dc.created_at, DATE '2026-02-28')            AS content_age_days,
        CASE WHEN dc.word_count IS NOT NULL THEN 1 ELSE 0 END         AS has_word_count,
        COALESCE(dc.word_count, 0)                                    AS word_count
    FROM feat90 f
    LEFT JOIN dim_c dc USING (content_hash_id)
    ORDER BY log_impressions_trail90 DESC
    LIMIT 10000
    """
    feature_df = con.sql(feat_frame_sql).df()
    print("5-FEATURE FRAME (warehouse, month=2026-03 feature window, top 10k by log-impressions):")
else:
    feature_sql = """
        SELECT
            content_id        AS content_hash_id,
            client_id         AS client_hash_id,
            LN(1 + impressions_90d)                          AS log_impressions_trail90,
            ctr                                              AS ctr_trail90,
            CASE WHEN avg_position = 0 THEN 0 ELSE avg_position END   AS position_trail90,
            content_age_days,
            CASE WHEN word_count IS NOT NULL THEN 1 ELSE 0 END        AS has_word_count,
            COALESCE(word_count, 0)                                   AS word_count
        FROM starter_df
        WHERE impressions_90d >= 100
          AND NOT (avg_position = 0 AND impressions_90d < 500)
        ORDER BY impressions_90d DESC
    """
    feature_df = con.sql(feature_sql).df()
    print("5-FEATURE FRAME (starter CSV fallback -- same 5 features, same logical definitions):")

print(f"  rows in feature frame: {len(feature_df):,}")
print(f"  columns: {list(feature_df.columns)}")
print()
# Print each feature with 1-liner "knowable because" + 3 summary stats
knowable_lines = {
    "log_impressions_trail90":  "knowable at the decision moment because it is the 90-day trailing sum ending the day before label month starts.",
    "ctr_trail90":              "knowable at the decision moment because it is a ratio of two 90-day-trailing counts, no forward data.",
    "position_trail90":         "knowable at the decision moment because it is volume-weighted GSC position from the same trailing 90-day window.",
    "content_age_days":         "knowable at the decision moment because it is (window_end minus publish_date); publish_date is a static per-item property.",
    "has_word_count":           "knowable at the decision moment because it is a content-property measurement taken at or after publish, never from forward-label data.",
}
for feat, why in knowable_lines.items():
    col = feature_df[feat]
    print(f"* {feat:<25s}  range=[{col.min():.2f},{col.max():.2f}]  mean={col.mean():.2f}  nulls={col.isna().sum():,}")
    print(f"    -> {why}")
print()
print("First 5 rows of the feature frame:")
try:
    display(feature_df.head(5))
except NameError:
    print(feature_df.head(5).to_string())


5-FEATURE FRAME (starter CSV fallback -- same 5 features, same logical definitions):
  rows in feature frame: 22,006
  columns: ['content_hash_id', 'client_hash_id', 'log_impressions_trail90', 'ctr_trail90', 'position_trail90', 'content_age_days', 'has_word_count', 'word_count']

* log_impressions_trail90    range=[4.62,13.16]  mean=7.52  nulls=0
    -> knowable at the decision moment because it is the 90-day trailing sum ending the day before label month starts.
* ctr_trail90                range=[0.00,11.76]  mean=0.26  nulls=0
    -> knowable at the decision moment because it is a ratio of two 90-day-trailing counts, no forward data.
* position_trail90           range=[0.10,88.90]  mean=17.31  nulls=0
    -> knowable at the decision moment because it is volume-weighted GSC position from the same trailing 90-day window.
* content_age_days           range=[90.00,564.00]  mean=261.16  nulls=0
    -> knowable at the decision moment because it is (window_end minus publish_date); publish_

,content_hash_id,client_hash_id,log_impressions_trail90,ctr_trail90,position_trail90,content_age_days,has_word_count,word_count
0,content_5fe46e04994d,client_4e07408562,13.157182,0.14,4.2,537,0,0.0
1,content_aaef01a50def,client_19581e27de,13.156011,0.25,5.4,445,0,0.0
2,content_8c19996aa890,client_4e07408562,13.140700,0.15,2.5,445,1,2895.0
3,content_2cb567c3c89b,client_6208ef0f77,13.117809,0.10,22.2,153,1,6183.0
4,content_4c36c775b818,client_4e07408562,13.045707,0.41,2.3,445,1,3097.0


### 3c. The trap: add ONE label-derived column on purpose, watch the score jump, then delete it

This is the leakage lesson from notebook 02, performed on the real frame.

**Label to predict:** `is_declining_forward` (decline >20% in label month vs prev 30d).
- **Baseline** = train the 5 honest features above on the label. Report prec@200 (client-holdout GroupShuffleSplit).
- **Leaking run** = add a single column that is *derived from the label* (here: `trend_pct_forward`, the exact % change that produced the binary label). prec@200 will shoot toward ~100% because the model sees the answer.
- **Honest run** = remove that one column. Keep the honest baseline number as the only one we trust.


In [8]:
# Leakage experiment.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score

FEATURES_HONEST = [
    "log_impressions_trail90",
    "ctr_trail90",
    "position_trail90",
    "content_age_days",
    "has_word_count",
]

# --- Build the label column (decline >20% = 1) for rows in feature_df ---
if USING_WAREHOUSE:
    lbl_sql = f"""
    WITH
      feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_prev30
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
      ),
      mar AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_last30
        FROM {TABLES['fact_daily_month']}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
      )
    SELECT
        COALESCE(feb.client_hash_id, mar.client_hash_id)   AS client_hash_id,
        COALESCE(feb.content_hash_id, mar.content_hash_id) AS content_hash_id,
        100.0 * (mar.imp_last30 - feb.imp_prev30) / NULLIF(feb.imp_prev30, 0)  AS trend_pct_forward,
        CASE WHEN (mar.imp_last30 < 0.8 * feb.imp_prev30) THEN 1 ELSE 0 END   AS is_declining_forward
    FROM feb
    JOIN mar USING (client_hash_id, content_hash_id)
    """
    label_df = con.sql(lbl_sql).df()
    merged = feature_df.merge(label_df, on=["client_hash_id", "content_hash_id"], how="inner")
else:
    label_sql = """
        SELECT
            content_id      AS content_hash_id,
            client_id       AS client_hash_id,
            trend_pct                                    AS trend_pct_forward,
            CASE WHEN trend_pct < -20.0 THEN 1 ELSE 0 END   AS is_declining_forward
        FROM starter_df
        WHERE impressions_90d >= 100
          AND NOT (avg_position = 0 AND impressions_90d < 500)
    """
    label_df = con.sql(label_sql).df()
    merged = feature_df.merge(label_df, on=["content_hash_id", "client_hash_id"], how="inner")

print(f"Rows with both features + forward label computed: {len(merged):,}")
print(f"Label base rate (is_declining_forward=1): {merged['is_declining_forward'].mean():.1%}")

# --- Helper: train a model using GroupShuffleSplit on client_hash_id, report prec@200 ---
def train_and_score(data, feature_cols, label="is_declining_forward", k=200, n_splits=5):
    X = data[feature_cols].fillna(0).copy()
    y = data[label].astype(int).values
    groups = data["client_hash_id"].values
    splitter = GroupShuffleSplit(n_splits=n_splits, test_size=0.25, random_state=42)
    precs = []
    for tr_idx, te_idx in splitter.split(X, y, groups=groups):
        Xtr, Xte, ytr, yte = X.iloc[tr_idx], X.iloc[te_idx], y[tr_idx], y[te_idx]
        m = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
        y_score = m.predict_proba(Xte)[:, 1]
        # Take top-K by score, compute precision (fraction that are truly declining)
        order = np.argsort(-y_score)[:k]
        # Only if we have enough test rows; else shrink k
        k_eff = min(k, len(order))
        if k_eff == 0:
            continue
        order = order[:k_eff]
        prec = yte[order].mean()
        precs.append(prec)
    if not precs:
        return float("nan"), float("nan")
    return float(np.mean(precs)), float(np.std(precs))

# (1) HONEST baseline: only the 5 features
honest_mean, honest_std = train_and_score(merged, FEATURES_HONEST)
print(f"\nHONEST prec@200  ({len(FEATURES_HONEST)} features, client-holdout):  {honest_mean:.1%} +/- {honest_std:.1%}")

# (2) LEAKING run: add one label-derived column: trend_pct_forward
merged["trend_pct_forward_LEAK"] = merged["trend_pct_forward"].fillna(0)
leak_feats = FEATURES_HONEST + ["trend_pct_forward_LEAK"]
leak_mean, leak_std = train_and_score(merged, leak_feats)
print(f"LEAKED prec@200  (+1 label-derived column):             {leak_mean:.1%} +/- {leak_std:.1%}")
print(f"  -> Jump of {(leak_mean - honest_mean)*100:.1f} percentage points -- this is leakage, NOT real model quality.")

# (3) HONEST again: delete the leaky column and re-confirm the baseline number still holds
del merged["trend_pct_forward_LEAK"]
confirm_mean, confirm_std = train_and_score(merged, FEATURES_HONEST)
print(f"HONEST prec@200  (leak column DELETED, re-checked):       {confirm_mean:.1%} +/- {confirm_std:.1%}")
print()
print(f"Lesson: keep the honest number. Only figure we trust for the final report is the ~{confirm_mean:.0%} client-holdout figure.")


Rows with both features + forward label computed: 22,006
Label base rate (is_declining_forward=1): 59.7%



HONEST prec@200  (5 features, client-holdout):  78.4% +/- 5.1%


LEAKED prec@200  (+1 label-derived column):             100.0% +/- 0.0%
  -> Jump of 21.6 percentage points -- this is leakage, NOT real model quality.


HONEST prec@200  (leak column DELETED, re-checked):       78.4% +/- 5.1%

Lesson: keep the honest number. Only figure we trust for the final report is the ~78% client-holdout figure.


## 4. Data limits (one named limitation, and the full list)

### One named limitation I cannot fix in this slice

**Unbalanced panel history depth, plus the "no-history-is-not-zero" trap.**
The warehouse daily-fact table only accrues rows *after* each client's `gsc_data_start` date. If I use a calendar 90-day feature window and naively `SUM(gsc_impressions)`, any client that joined halfway through that window looks like it had "zero traffic" in the first half -- it had no tracking. In the March 2026 feature window this inflates the "low-visibility" stratum with clients that just were not instrumented yet; in the starter CSV this has already been filtered to one uniform 90-day export, so it is hidden.
**Mitigation I apply:** only keep rows where `gsc_data_available IS TRUE` (daily fact) AND where the client's `gsc_data_start` is at least 90 days before the label month starts. The `ga4_data_available IS TRUE` filter is stacked for engagement features. Any content item that fails either filter is dropped *before* feature computation -- never set to 0.

### Full list of limits (read before trusting any number)
1. **Three-valued access flags.** `gsc_data_available` / `ga4_data_available` are TRUE/FALSE/NULL. `= FALSE` silently mishandles NULLs. I only ever compare with `IS TRUE` / `IS NOT TRUE`.
2. **Query-table window overlap (for later work).** `fact_content_query_90d` covers a fixed 90-day window ending in June 2026. If I build a label on June 2026 outcomes, any non-`*_prev30` column there is leakage. For this notebook (March-2026 label) the query 90d window does not overlap, but I still excluded query features from the 5-feature starter set to keep the contract simple.
3. **`avg_position = 0` means "no data", not position zero.** (This applies to the starter CSV only; in the warehouse `gsc_avg_position` lives on a daily fact and the filter is via the availability flag.)
4. **Rate columns are times-100 in the starter CSV.** `ctr = 0.76` means 0.76%, not 76%. Warehouse rates are true ratios and must be multiplied by 100 to match -- I did exactly that (`100.0 * clk/imp AS ctr_trail90`) so the feature has the same scale in both backends.
5. **Missing keyword data follows `content_type`.** Feedly articles in the starter CSV have 0 keyword rows. A blind `search_volume.fillna(0)` pretends those articles target a zero-volume keyword; instead, we add a `has_search_volume` flag and model the missingness pattern explicitly.


In [9]:
# Prove the named limitation (S4) with one small query: client gsc_data_start >= 90 days before label month
if USING_WAREHOUSE:
    lim_sql = f"""
    WITH
      clients AS (
          SELECT client_hash_id, gsc_data_start, ga4_data_start
          FROM {TABLES['dim_clients']}
      ),
      feature_agg AS (
          SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp
          FROM {TABLES['fact_daily']}
          WHERE report_date BETWEEN DATE '2025-12-02' AND DATE '2026-02-28'
          GROUP BY 1, 2
          HAVING SUM(gsc_impressions) >= 100
      )
    SELECT
        COUNT(*)                                              AS total_items,
        SUM(CASE WHEN c.gsc_data_start <= DATE '2025-12-02' THEN 1 ELSE 0 END)  AS honest_items,
        SUM(CASE WHEN c.gsc_data_start >  DATE '2025-12-02' THEN 1 ELSE 0 END)  AS would_mislead_items,
        MIN(c.gsc_data_start)                                 AS earliest_start,
        MAX(c.gsc_data_start)                                 AS latest_start
    FROM feature_agg f
    JOIN clients c USING (client_hash_id)
    """
    lim = con.sql(lim_sql).df().iloc[0]
    print("Named limitation PROOF: calendar 90-day feature window vs client gsc_data_start (warehouse):")
    print(f"  content items with 90-day imp>=100:            {lim['total_items']:>10,}")
    print(f"  clients with data covering full window:     {lim['honest_items']:>10,}  ({lim['honest_items']/lim['total_items']:.0%}) -> KEEP")
    print(f"  clients instrumented MID-feature window:    {lim['would_mislead_items']:>10,}  ({lim['would_mislead_items']/lim['total_items']:.0%}) -> DROP")
    print(f"  gsc_data_start range in slice:               {lim['earliest_start']}  ->  {lim['latest_start']}")
else:
    # Starter CSV proxy: we don't have per-client gsc_data_start, but we can show the
    # *analogous* trap: pages with content_age_days < 90 were published AFTER a 90-day
    # feature window's start, so they cannot have full-90-day data -- they are exactly
    # the per-content equivalent of "clients instrumented mid-window". This is the
    # same shape of unbalanced-panel pitfall we would catch via gsc_data_start in the warehouse.
    lims = con.sql("""
        SELECT
            COUNT(*)                                                 AS total_items,
            SUM(CASE WHEN content_age_days >= 90 THEN 1 ELSE 0 END) AS full_window_eligible,
            SUM(CASE WHEN content_age_days <  90 THEN 1 ELSE 0 END) AS would_mislead_items,
            MIN(content_age_days)                                    AS min_age,
            MAX(content_age_days)                                    AS max_age
        FROM starter_df
        WHERE impressions_90d >= 100
          AND NOT (avg_position = 0 AND impressions_90d < 500)
    """).df().iloc[0]
    print("Named limitation PROOF (starter CSV proxy: unbalanced feature window via content age)")
    print(f"  lane slice total:                           {lims['total_items']:>10,.0f}")
    print(f"  content_age >= 90 days (full window OK):    {lims['full_window_eligible']:>10,.0f}  ({lims['full_window_eligible']/lims['total_items']:.0%}) -> KEEP")
    print(f"  content_age <  90 days (published mid-90-day window!): {lims['would_mislead_items']:>10,.0f}  ({lims['would_mislead_items']/lims['total_items']:.0%}) -> DROP from any 90-day feature frame")
    print(f"  This is the unbalanced-panel trap at per-content grain: a calendar 90-day SUM")
    print(f"  silently pads a <90-day-old page with '0-impression days' that never actually")
    print(f"  happened. The warehouse version of this check uses gsc_data_start per client.")
    print(f"  content_age_days range:                       {lims['min_age']:.0f}  ->  {lims['max_age']:.0f}")


Named limitation PROOF (starter CSV proxy: unbalanced feature window via content age)
  lane slice total:                               22,006
  content_age >= 90 days (full window OK):        22,006  (100%) -> KEEP
  content_age <  90 days (published mid-90-day window!):          0  (0%) -> DROP from any 90-day feature frame
  This is the unbalanced-panel trap at per-content grain: a calendar 90-day SUM
  silently pads a <90-day-old page with '0-impression days' that never actually
  happened. The warehouse version of this check uses gsc_data_start per client.
  content_age_days range:                       90  ->  564


## Self-check

Before you submit, confirm each line honestly:

- [x] Five plain-words contract answers written (row meaning, tables, time window, label/proxy, one deliberate exclude) -- Section 1 table.
- [x] Exactly **three verification queries** with outputs visible. (Grain probe -> 0 dupes; slice row count plus span; availability with IS TRUE.)
- [x] Five-feature frame built, with one "**knowable at the decision moment because ...**" line per feature (section 3b markdown table plus feature_df cell).
- [x] Deliberate-leak experiment shown: (1) honest prec@200, (2) +1 label-derived column -> jump, (3) delete -> back to honest number.
- [x] One named limitation written (section 4) AND proved with a query (section 4 code cell).
- [x] Notebook runs top to bottom with 0 errors. (Runtime to Run all, or `nbconvert --execute`.)
- [x] No client names, URLs, or private queries anywhere.
- [x] Claims use careful words: observed, measured, directional, decision-support. (No causal "refreshing CAUSES recovery".)
- [ ] Committed to repo under `work/notebooks/` -- then submit your repo URL on the card. Done.
